In [2]:
import pandas as pd
import random as rand
import openpyxl

# Function to simulate rolling dice
def dice(amount:int,sides:int):
    roll = 0
    for i in range(0,amount):
        die = rand.randint(1,sides)
        roll = roll + die
    return(roll)

# Function to find a specific character in a string
# Returns location or None if that character is not in the string
def string_finder(string:str,char:str):
    for pos in range(len(string)):
        if string[pos] == char:
            return(pos)
        elif pos == len(string):
            return(False)  

# Function to find the row index of a die roll within a table of ranges
# Returns column index within given column constraints    
def range_finder(roll_result:int,table_name:str,column_index:int,start_index:int,end_index:int):
    odds_list = table_name.iloc[start_index:end_index,column_index].tolist()
    for row in range(len(odds_list)):
        odds = odds_list[row]
        hyphen_loc = string_finder(odds,'-')
        
        if hyphen_loc == None:
            low = int(odds)
            high = low
        else:
            low = int(odds[0:hyphen_loc])
            high = int(odds[hyphen_loc+1:len(odds)])

        if low <= roll_result <= high:
            return row
      

In [3]:
CR = rand.randint(1,20)
CR

10

In [4]:
# Obtain individual encounter tables from Excel sheet
indv_tables = pd.read_excel('C:/Users/Anderson/Documents/DnD_data.xlsx',sheet_name='Individual Treasure Tables')

# Find indicies where all values are empty (these separate the tables on the excel sheet)
sep = indv_tables[indv_tables.isnull().all(axis=1)].index.tolist()

# Determine the range of indicies to pull from the table based the encounter's Challenge Rating
if 0 <= CR <= 4:
    start = 0
    end = sep[0]
elif 5 <= CR <= 10:
    start = sep[0]+1
    end = sep[1]
elif 11 <= CR <= 16:
    start = sep[1]+1
    end = sep[2]
elif CR > 16:
    start = sep[2]+1
    end = len(indv_tables)
else:
    print("Error: Please enter a positive CR.")

# Extract appropriate indicies from encounter table
loot_table = indv_tables.iloc[start:end].reset_index(drop=True)
loot_table

,d100,cp,sp,ep,gp,pp
0,1-30,4d6 x 100,NaN,1d6 x 10,NaN,NaN
1,31-60,NaN,6d6 x 10,NaN,2d6 x 10,NaN
2,61-70,NaN,NaN,3d6 x 10,2d6 x 10,NaN
3,71-95,NaN,NaN,NaN,4d6 x 10,NaN
4,96-100,NaN,NaN,NaN,2d6 x 10,3d6


In [5]:
# Roll a d100 to determine which 
loot_roll = dice(1,100)
loot_index = range_finder(loot_roll,loot_table,0,0,len(loot_table))
print("roll:",loot_roll)
print("row:",loot_index)


roll: 16
row: 0


In [138]:
coin_table = loot_table.loc[loot_index,'cp':].dropna()
coin_list = [coin_table[i]+'~'+coin_table.index[i] for i in range(len(coin_table))]
print(coin_list)

['6d6 x 10~sp', '2d6 x 10~gp']


In [ ]:
coin_list = loot_table.loc[loot_index,'cp':].dropna().tolist()
for i in range(len(coin_list)):
    value = coin_list[i]
    denom = loot_table.columns[i+1]
    
    print(value,denom)

4d6 x 100 cp
1d6 x 10 sp


In [159]:
# Break each value into its denominational components (may just be one)
for coin in coin_list:
    # Find location of 'd' within each string
    d_loc = string_finder(coin,'d')

    # Extract the amount of dice to be rolled
    amt = int(coin[0:d_loc])

    # Use the location of the space (if it exists) to extract the # of sides and multiplier
    space_loc = string_finder(coin,' ')
    den_loc = string_finder(coin,'~')
    if space_loc == None: #func string_finder returns False if that character does not exist
        sides = int(coin[d_loc+1:den_loc])
        mult = 1
    else:
        sides = int(coin[d_loc+1:space_loc])
        mult = int(coin[space_loc+3:den_loc])

    # Roll the amount of each denomination that appears
    value = dice(amt,sides)*mult
        
    # extract the denomination of the coin (cp,sp,etc) from the string
    denom = coin[den_loc+1:len(coin)]

    # display loot for each type of coin
    print(str(value)+' '+denom)

220 sp
70 gp
